In [2]:
import sys
from pathlib import Path

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.metrics import score_tokens

reference = "学习ASR模型"
hypothesis = "学习LLM模型"

#CER：按字符
ref_chars = list(reference)
hyp_chars = list(hypothesis)

cer_result = score_tokens(ref_chars,hyp_chars)

#MER：先动手构造混合token
ref_mixed = ["学", "习", "ASR", "模", "型"]
hyp_mixed = ["学", "习", "LLM", "模", "型"]

mer_result = score_tokens(ref_mixed, hyp_mixed)

print("CER tokens:", ref_chars)
print("CER:", cer_result["error_rate"])
print("CER distance:", cer_result["distance"])

print("\nMER tokens:", ref_mixed)
print("MER:", mer_result["error_rate"])
print("MER distance:", mer_result["distance"])

assert cer_result["distance"] == 3
assert mer_result["distance"] == 1
assert abs(cer_result["error_rate"] - 3/7) < 1e-8
assert abs(mer_result["error_rate"] - 1/5) < 1e-8

print("\n模块1实验通过")

CER tokens: ['学', '习', 'A', 'S', 'R', '模', '型']
CER: 0.42857142857142855
CER distance: 3

MER tokens: ['学', '习', 'ASR', '模', '型']
MER: 0.2
MER distance: 1

模块1实验通过


In [2]:
#检验两个函数是否符合预期
import sys
from pathlib import Path

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.normalize import normalize_score, mixed_tokens

#1. NFKC + 小写 + 标点
text1 = "  Ｃ＋＋，GPT-４！  "
print("规范化：", normalize_score(text1))

# 2. 中文、数字、英文
text2 = "我在2026年学习Python"
print("混合分词:", mixed_tokens(text2))

# 3. 技术表达
text3 = "C++、C#、GPT-4、v1.2.3、3.14"
print("技术术语：", mixed_tokens(text3))

# 4. 未知有效符号
text4 = "A🙂B"
print("未知符号：", mixed_tokens(text4))

#验收断言
assert normalize_score(text1) == "c++ gpt-4"

assert mixed_tokens(text2) == [
    "我", "在", "2026", "年",
    "学", "习", "python"
]

assert mixed_tokens(text3) == [
    "c++", "c#", "gpt-4", "v1.2.3", "3.14"
]

assert mixed_tokens(text4) == [
    "a", "🙂", "b"
]

print("\n模块2实验通过")

规范化： c++ gpt-4
混合分词: ['我', '在', '2026', '年', '学', '习', 'python']
技术术语： ['c++', 'c#', 'gpt-4', 'v1.2.3', '3.14']
未知符号： ['a', '🙂', 'b']

模块2实验通过


In [3]:
#准备公共评分函数
import importlib
import src.normalize as normalize_module
import src.metrics as metrics_module
from pathlib import Path

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

# Jupyter 中重新加载本地模块
importlib.reload(normalize_module)
importlib.reload(metrics_module)

from src.normalize import normalize_score, mixed_tokens
from src.metrics import score_tokens, score_corpus

def cer_tokens(text):
    normalized = normalize_score(text)
    return list("".join(normalized.split()))

def english_tokens(text):
    normalized = normalize_score(text)
    return normalized.split()

def score_text_pair(reference, hypothesis, tokenizer):
    return score_tokens(
        tokenizer(reference),
        tokenizer(hypothesis),
    )

In [4]:
#六组测试
# ========== 测试1：统一规范化 ==========
ref = " Ｐｙｔｈｏｎ，ASR！ "
hyp = "python asr"

assert normalize_score(ref) == normalize_score(hyp)

cer = score_text_pair(ref, hyp, cer_tokens)
mer = score_text_pair(ref, hyp, mixed_tokens)

assert cer["distance"] == 0
assert mer["distance"] == 0

print("测试1通过：统一规范化")


# ========== 测试2：CER / MER 差异 ==========
ref = "学习ASR模型"
hyp = "学习LLM模型"

cer = score_text_pair(ref, hyp, cer_tokens)
mer = score_text_pair(ref, hyp, mixed_tokens)

assert cer["N"] == 7
assert cer["distance"] == 3
assert mer["N"] == 5
assert mer["distance"] == 1

assert abs(cer["error_rate"] - 3/7) < 1e-8
assert abs(mer["error_rate"] - 1/5) < 1e-8

print("测试2通过：CER / MER 差异")

# ========== 测试3：数字不自动合并 ==========
ref = "价格3元"
hyp = "价格三元"

assert normalize_score(ref) != normalize_score(hyp)

mer = score_text_pair(ref, hyp, mixed_tokens)

assert mer["N"] == 4
assert mer["distance"] == 1
assert abs(mer["error_rate"] - 0.25) < 1e-8

print("测试3通过：数字保守计分")


# ========== 测试4：特殊符号 ==========
ref = "C++"
hyp = "C"

assert mixed_tokens(ref) == ["c++"]
assert mixed_tokens(hyp) == ["c"]

cer = score_text_pair(ref, hyp, cer_tokens)
mer = score_text_pair(ref, hyp, mixed_tokens)

assert cer["distance"] == 2
assert mer["distance"] == 1

# 检查技术表达与未知有效符号
assert mixed_tokens("GPT-4") == ["gpt-4"]
assert mixed_tokens("C#") == ["c#"]
assert mixed_tokens("A🙂B") == ["a", "🙂", "b"]

# 未知符号发生变化，也应产生错误
emoji = score_text_pair(
    "A🙂B", "A🙃B", mixed_tokens
)
assert emoji["distance"] == 1

print("测试4通过：特殊符号不丢失")


# ========== 测试5：English WER ==========
ref = "I like Python"
hyp = "i love python"

english = score_text_pair(
    ref, hyp, english_tokens
)

assert english["N"] == 3
assert english["S"] == 1
assert english["D"] == 0
assert english["I"] == 0
assert abs(english["error_rate"] - 1/3) < 1e-8

print("测试5通过：English WER")


# ========== 测试6：语料级聚合 ==========
references = [
    "hello",
    "one two three four five six seven eight nine",
    "",
]

hypotheses = [
    "",
    "one two three four five six seven eight nine",
    "extra",
]

pairs = [
    (mixed_tokens(ref), mixed_tokens(hyp))
    for ref, hyp in zip(references, hypotheses)
]

corpus = score_corpus(pairs)

assert corpus["N"] == 10
assert corpus["distance"] == 2
assert corpus["D"] == 1
assert corpus["I"] == 1
assert abs(corpus["error_rate"] - 0.2) < 1e-8

print("测试6通过：语料级聚合")
print("\n六组指标测试全部通过")

测试1通过：统一规范化
测试2通过：CER / MER 差异
测试3通过：数字保守计分
测试4通过：特殊符号不丢失
测试5通过：English WER
测试6通过：语料级聚合

六组指标测试全部通过
